In [1]:
import tensorflow as tf
import numpy as np
import os
import logging
import string
import random
import yaml
import ast
from datetime import datetime
from tqdm.notebook import tqdm

# 可视化库
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D

from dimenet.model.dimenet import DimeNet
from dimenet.model.dimenet_pp import DimeNetPP
from dimenet.model.activations import swish
from dimenet.training.trainer import Trainer
from dimenet.training.metrics import Metrics
from dimenet.training.data_container import DataContainer
from dimenet.training.data_provider import DataProvider

# 设置matplotlib中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

/opt/anaconda3/envs/lewis_acid_env/lib/python3.9/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
# Set up logger
logger = logging.getLogger()
logger.handlers = []
ch = logging.StreamHandler()
formatter = logging.Formatter(
        fmt='%(asctime)s (%(levelname)s): %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S')
ch.setFormatter(formatter)
logger.addHandler(ch)
logger.setLevel('INFO')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
tf.get_logger().setLevel('WARN')
tf.autograph.set_verbosity(2)

### Load config

In [3]:
# 使用您训练时使用的配置文件
with open('config_pp_minimal.yaml', 'r') as c:
    config = yaml.safe_load(c)
    
print("Configuration loaded:")
print(f"Model: {config['model_name']}")
print(f"Targets: {config['targets']}")
print(f"Dataset: {config['dataset']}")
print(f"Minimal output: {config.get('minimal_output', False)}")
print(f"Freeze features: {config.get('freeze_features', False)}")

Configuration loaded:
Model: dimenet++
Targets: ['fia_gas-DSDBLYP']
Dataset: data/fia49k_processed.npz
Minimal output: True
Freeze features: True


In [4]:
model_name = config['model_name']

if model_name == "dimenet":
    num_bilinear = config['num_bilinear']
elif model_name == "dimenet++":
    out_emb_size = config['out_emb_size']
    int_emb_size = config['int_emb_size']
    basis_emb_size = config['basis_emb_size']
    extensive = config['extensive']
else:
    raise ValueError(f"Unknown model name: '{model_name}'")
    
emb_size = config['emb_size']
num_blocks = config['num_blocks']

num_spherical = config['num_spherical']
num_radial = config['num_radial']
output_init = config['output_init']

cutoff = config['cutoff']
envelope_exponent = config['envelope_exponent']

num_before_skip = config['num_before_skip']
num_after_skip = config['num_after_skip']
num_dense_output = config['num_dense_output']

num_train = config['num_train']
num_valid = config['num_valid']
data_seed = config['data_seed']
dataset_path = config['dataset']

batch_size = config['batch_size']

# 新增
minimal_output = config.get('minimal_output', False)
freeze_features = config.get('freeze_features', False)
pretrained_weights_path = config.get('pretrained_weights_path', None)

#####################################################################
# Change this if you want to predict a different target, e.g. to ['U0']
# (but don't forget to change output_init as well)
targets = config['targets']
#####################################################################

### Load dataset

In [5]:
print(f"Loading dataset from: {dataset_path}")
data_container = DataContainer(dataset_path, cutoff=cutoff, target_keys=targets)

# Initialize DataProvider (splits dataset into training, validation and test set based on data_seed)
data_provider = DataProvider(data_container, num_train, num_valid, batch_size,
                             seed=data_seed, randomized=True)

print(f"Dataset loaded successfully!")
print(f"Training samples: {data_provider.nsamples['train']}")
print(f"Test samples: {data_provider.nsamples['test']}")

print(f"\n=== 数据划分确认 ===")
print(f"✅ 我们正在预测测试集，包含 {data_provider.nsamples['test']} 个样本")
print(f"✅ 这些样本在训练过程中从未被模型见过")
print(f"✅ 数据划分使用随机种子: {data_seed}")

# Initialize datasets
dataset = data_provider.get_dataset('test').prefetch(tf.data.experimental.AUTOTUNE)
dataset_iter = iter(dataset)
print("Test dataset iterator created successfully!")

Loading dataset from: data/fia49k_processed.npz
数据加载完成，使用原始目标值
Dataset loaded successfully!
Training samples: 1600
Test samples: 315

=== 数据划分确认 ===
✅ 我们正在预测测试集，包含 315 个样本
✅ 这些样本在训练过程中从未被模型见过
✅ 数据划分使用随机种子: 42
Metal device set to: Apple M1

systemMemory: 8.00 GB
maxCacheSize: 2.67 GB

Test dataset iterator created successfully!


2025-10-16 21:09:14.024684: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


### Initialize model

In [6]:
if model_name == "dimenet":
    model = DimeNet(
            emb_size=emb_size, num_blocks=num_blocks, num_bilinear=num_bilinear,
            num_spherical=num_spherical, num_radial=num_radial,
            cutoff=cutoff, envelope_exponent=envelope_exponent,
            num_before_skip=num_before_skip, num_after_skip=num_after_skip,
            num_dense_output=num_dense_output, num_targets=len(targets),
            activation=swish, output_init=output_init)
elif model_name == "dimenet++":
    print("Initializing DimeNet++ model with parameters:")
    print(f"  emb_size: {emb_size}")
    print(f"  out_emb_size: {out_emb_size}")
    print(f"  int_emb_size: {int_emb_size}")
    print(f"  basis_emb_size: {basis_emb_size}")
    print(f"  num_blocks: {num_blocks}")
    print(f"  minimal_output: {minimal_output}")
    print(f"  freeze_features: {freeze_features}")
    print(f"  pretrained_weights_path: {pretrained_weights_path}")
    
    model = DimeNetPP(
            emb_size=emb_size, out_emb_size=out_emb_size,
            int_emb_size=int_emb_size, basis_emb_size=basis_emb_size,
            num_blocks=num_blocks, num_spherical=num_spherical, num_radial=num_radial,
            cutoff=cutoff, envelope_exponent=envelope_exponent,
            num_before_skip=num_before_skip, num_after_skip=num_after_skip,
            num_dense_output=num_dense_output, num_targets=len(targets),
            activation=swish, extensive=extensive, output_init=output_init,
            minimal_output=minimal_output,
            freeze_features=freeze_features,
            pretrained_weights_path=pretrained_weights_path)
else:
    raise ValueError(f"Unknown model name: '{model_name}'")

Initializing DimeNet++ model with parameters:
  emb_size: 128
  out_emb_size: 256
  int_emb_size: 64
  basis_emb_size: 8
  num_blocks: 4
  minimal_output: True
  freeze_features: True
  pretrained_weights_path: pretrained/dimenet_pp/U/ckpt
冻结特征层完成，输出层保持可训练


### Initialize trainer

In [7]:
trainer = Trainer(model, early_stopping_patience=0)

### Load weights from model at best step

In [8]:
#####################################################################
# Load the trained model from your own training run
# 使用您训练了5000步保存的权重
directory = "20251009_214042_xo8fhxiW_fia49k_processed.npz_fia_gas-DSDBLYP_FIA minimal output"
best_ckpt_file = os.path.join(directory, 'best', 'ckpt')
#####################################################################
# 注释掉预训练模型路径
# directory = f"pretrained/dimenet_pp/U0"
# best_ckpt_file = os.path.join(directory, 'ckpt')
#####################################################################

print(f"Loading weights from: {best_ckpt_file}")
model.load_weights(best_ckpt_file)
print("Weights loaded successfully!")

Loading weights from: 20251009_214042_xo8fhxiW_fia49k_processed.npz_fia_gas-DSDBLYP_FIA minimal output/best/ckpt


NotFoundError: Unsuccessful TensorSliceReader constructor: Failed to find any matching files for 20251009_214042_xo8fhxiW_fia49k_processed.npz_fia_gas-DSDBLYP_FIA minimal output/best/ckpt

In [ ]:
### 验证模型权重加载
# 检查模型是否已正确加载权重
print("验证模型权重加载状态...")
print(f"模型可训练参数数量: {len(model.trainable_variables)}")

# 显示一些关键层的权重信息
for i, var in enumerate(model.trainable_variables[:5]):  # 只显示前5个变量
    print(f"参数 {i+1}: {var.name}, 形状: {var.shape}, 数据类型: {var.dtype}")

print("权重验证完成!")


### Prediction loop

In [ ]:
# Initialize aggregates
metrics = Metrics('val', targets)
preds_total = np.zeros([data_provider.nsamples['test'], len(targets)], dtype=np.float32)
targets_total = np.zeros([data_provider.nsamples['test'], len(targets)], dtype=np.float32)  # 保存真实值

In [ ]:
steps_per_epoch = int(np.ceil(data_provider.nsamples['test'] / batch_size))

for step in tqdm(range(steps_per_epoch)):
    # 获取预测值和真实值
    inputs, targets_batch = next(dataset_iter)
    preds = model(inputs, training=False)
    
    # 更新指标
    mae = tf.reduce_mean(tf.abs(targets_batch - preds), axis=0)
    mean_mae = tf.reduce_mean(mae)
    loss = mean_mae
    nsamples = tf.shape(preds)[0]
    metrics.update_state(loss, mean_mae, mae, nsamples)
    
    # 保存预测值和真实值
    batch_start = step * batch_size
    batch_end = min((step + 1) * batch_size, data_provider.nsamples['test'])
    preds_total[batch_start:batch_end] = preds.numpy()
    targets_total[batch_start:batch_end] = targets_batch.numpy()

In [ ]:
print(f"\n=== 测试集预测结果 ===")
print(f"✅ 测试集样本数: {data_provider.nsamples['test']}")
print(f"✅ 目标属性: {','.join(targets)}")
print(f"✅ 测试集MAE: {metrics.mean_mae:.6f}")
print(f"✅ 测试集logMAE: {metrics.mean_log_mae:.6f}")

# 显示一些预测样本
print(f"\n=== 测试集预测样本 (前10个) ===")
for i in range(min(10, len(preds_total))):
    print(f"测试样本 {i+1}: {preds_total[i][0]:.6f}")

# 保存预测结果
output_file = f"test_predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.npz"
np.savez(output_file, 
         predictions=preds_total, 
         targets=targets,
         mae=metrics.mean_mae,
         log_mae=metrics.mean_log_mae,
         test_samples=data_provider.nsamples['test'],
         data_seed=data_seed)
print(f"\n✅ 测试集预测结果已保存到: {output_file}")
print(f"✅ 这是真正的测试集预测，模型在训练时从未见过这些样本")

In [ ]:
### 可视化预测结果
print("开始生成可视化图表...")

# 提取预测值和真实值
y_true = targets_total[:, 0]  # 真实值
y_pred = preds_total[:, 0]    # 预测值
errors = y_pred - y_true      # 预测误差

# 计算额外的评估指标
mae_sklearn = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
rmse = np.sqrt(np.mean(errors**2))

print(f"详细评估指标:")
print(f"MAE: {mae_sklearn:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²: {r2:.6f}")


In [ ]:
# Create visualization charts
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Test Set Prediction Results Visualization', fontsize=16, fontweight='bold')

# 1. Predicted vs True values scatter plot
ax1 = axes[0, 0]
ax1.scatter(y_true, y_pred, alpha=0.6, s=20)
min_val = min(y_true.min(), y_pred.min())
max_val = max(y_true.max(), y_pred.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction Line')
ax1.set_xlabel('True Values')
ax1.set_ylabel('Predicted Values')
ax1.set_title(f'Predicted vs True Values\nR² = {r2:.4f}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Prediction error distribution histogram
ax2 = axes[0, 1]
ax2.hist(errors, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error Line')
ax2.set_xlabel('Prediction Error (Predicted - True)')
ax2.set_ylabel('Frequency')
ax2.set_title(f'Prediction Error Distribution\nMAE = {mae_sklearn:.4f}')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Residual plot
ax3 = axes[1, 0]
ax3.scatter(y_pred, errors, alpha=0.6, s=20)
ax3.axhline(0, color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('Predicted Values')
ax3.set_ylabel('Residuals (Predicted - True)')
ax3.set_title('Residual Plot')
ax3.grid(True, alpha=0.3)

# 4. Sample comparison (first 20 samples)
ax4 = axes[1, 1]
sample_indices = np.arange(min(20, len(y_true)))
x_pos = np.arange(len(sample_indices))
width = 0.35

ax4.bar(x_pos - width/2, y_true[sample_indices], width, label='True Values', alpha=0.8)
ax4.bar(x_pos + width/2, y_pred[sample_indices], width, label='Predicted Values', alpha=0.8)
ax4.set_xlabel('Sample Index')
ax4.set_ylabel('Target Value')
ax4.set_title('First 20 Test Samples Comparison')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization charts generated successfully!")


In [ ]:
### 详细预测分析
print("=== 详细预测分析 ===")

# 创建DataFrame用于分析
df_results = pd.DataFrame({
    '真实值': y_true,
    '预测值': y_pred,
    '误差': errors,
    '绝对误差': np.abs(errors),
    '相对误差': np.abs(errors) / (np.abs(y_true) + 1e-8) * 100
})

print(f"预测统计信息:")
print(f"真实值范围: [{y_true.min():.4f}, {y_true.max():.4f}]")
print(f"预测值范围: [{y_pred.min():.4f}, {y_pred.max():.4f}]")
print(f"误差范围: [{errors.min():.4f}, {errors.max():.4f}]")
print(f"平均绝对误差: {df_results['绝对误差'].mean():.4f}")
print(f"误差标准差: {errors.std():.4f}")
print(f"最大绝对误差: {df_results['绝对误差'].max():.4f}")
print(f"平均相对误差: {df_results['相对误差'].mean():.2f}%")

# 显示预测最好的和最差的样本
print(f"\n预测最好的5个样本 (绝对误差最小):")
best_samples = df_results.nsmallest(5, '绝对误差')
for idx, row in best_samples.iterrows():
    print(f"样本 {idx}: 真实值={row['真实值']:.4f}, 预测值={row['预测值']:.4f}, 误差={row['误差']:.4f}")

print(f"\n预测最差的5个样本 (绝对误差最大):")
worst_samples = df_results.nlargest(5, '绝对误差')
for idx, row in worst_samples.iterrows():
    print(f"样本 {idx}: 真实值={row['真实值']:.4f}, 预测值={row['预测值']:.4f}, 误差={row['误差']:.4f}")

# 保存详细结果
detailed_results_file = f"detailed_test_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_results.to_csv(detailed_results_file, index=True)
print(f"\n详细预测结果已保存到: {detailed_results_file}")


In [ ]:
### 可解释性分析类
class DimeNetExplainability:
    """DimeNet++ 可解释性分析器 - 基于RBF注意力机制"""
    
    def __init__(self, model, data_container):
        self.model = model
        self.data_container = data_container
    
    def analyze_molecule(self, inputs, target_class=0):
        """分析单个分子的可解释性"""
        
        # 确保输入数据格式正确
        if 'R' not in inputs or 'Z' not in inputs:
            raise ValueError("输入数据缺少必要的键 'R' 或 'Z'")
        
        # 1. 提取RBF注意力权重（无需梯度计算）
        rbf_attention = self._extract_rbf_attention(inputs)
        
        # 2. 计算原子坐标梯度
        atomic_gradients = self._compute_atomic_gradients(inputs, target_class)
        
        # 3. 计算键重要性
        bond_importance = self._compute_bond_importance(inputs, target_class, rbf_attention)
        
        # 4. 分子级分析
        molecular_contribution = self._analyze_molecular_contribution(inputs, target_class)
        
        return {
            'rbf_attention': rbf_attention,
            'atomic_gradients': atomic_gradients,
            'bond_importance': bond_importance,
            'molecular_contribution': molecular_contribution,
            'coordinates': inputs['R'],
            'atomic_numbers': inputs['Z'],
            'bond_indices': (inputs['idnb_i'], inputs['idnb_j']),
            'distances': self.model.calculate_interatomic_distances(
                inputs['R'], inputs['idnb_i'], inputs['idnb_j'])
        }
    
    def _extract_rbf_attention(self, inputs):
        """提取RBF注意力权重"""
        # 将numpy数组转换为TensorFlow张量
        R_tensor = tf.constant(inputs['R'], dtype=tf.float32)
        
        # 计算距离
        Dij = self.model.calculate_interatomic_distances(
            R_tensor, inputs['idnb_i'], inputs['idnb_j'])
        
        # 通过RBF层
        rbf = self.model.rbf_layer(Dij)
        
        # 通过输出层的RBF门控（使用第一个输出块）
        rbf_gate = self.model.output_blocks[0].dense_rbf(rbf)
        
        # 对RBF门控进行聚合，得到每个键的重要性
        rbf_attention = tf.reduce_sum(tf.abs(rbf_gate), axis=-1)
        
        return rbf_attention  # 这就是RBF注意力权重
    
    def _compute_atomic_gradients(self, inputs, target_class):
        """计算原子坐标梯度"""
        # 将numpy数组转换为TensorFlow张量
        R_tensor = tf.constant(inputs['R'], dtype=tf.float32)
        
        with tf.GradientTape() as tape:
            tape.watch(R_tensor)
            # 创建新的inputs字典，使用张量版本的R
            inputs_tensor = inputs.copy()
            inputs_tensor['R'] = R_tensor
            predictions = self.model(inputs_tensor, training=False)
            target_pred = predictions[:, target_class]
        
        gradients = tape.gradient(target_pred, R_tensor)
        return gradients
    
    def _compute_bond_importance(self, inputs, target_class, rbf_attention):
        """计算键重要性 - 结合RBF注意力和梯度"""
        # 将numpy数组转换为TensorFlow张量
        R_tensor = tf.constant(inputs['R'], dtype=tf.float32)
        
        # 计算距离
        Dij = self.model.calculate_interatomic_distances(
            R_tensor, inputs['idnb_i'], inputs['idnb_j'])
        
        # 计算距离梯度
        with tf.GradientTape() as tape:
            tape.watch(R_tensor)
            # 创建新的inputs字典，使用张量版本的R
            inputs_tensor = inputs.copy()
            inputs_tensor['R'] = R_tensor
            predictions = self.model(inputs_tensor, training=False)
            target_pred = predictions[:, target_class]
        
        gradients = tape.gradient(target_pred, R_tensor)
        
        # 计算每个键的梯度
        Ri = tf.gather(R_tensor, inputs['idnb_i'])
        Rj = tf.gather(R_tensor, inputs['idnb_j'])
        Gi = tf.gather(gradients, inputs['idnb_i'])
        Gj = tf.gather(gradients, inputs['idnb_j'])
        
        # 键梯度 = 两个原子梯度的组合
        bond_gradients = tf.norm(Gi - Gj, axis=-1)
        
        # 结合RBF注意力权重
        bond_importance = tf.abs(bond_gradients) * tf.abs(rbf_attention)
        
        return {
            'importance': bond_importance,
            'distances': Dij,
            'gradients': bond_gradients,
            'rbf_attention': rbf_attention
        }
    
    def _analyze_molecular_contribution(self, inputs, target_class):
        """分析分子级贡献"""
        predictions = self.model(inputs, training=False)
        return predictions[:, target_class]
    
    def visualize_explanations(self, analysis_results, molecule_index=0, 
                              save_path=None, show_plot=True):
        """可视化可解释性结果"""
        coords = analysis_results['coordinates']
        atomic_nums = analysis_results['atomic_numbers']
        rbf_attention = analysis_results['rbf_attention']
        atomic_grads = analysis_results['atomic_gradients']
        bond_importance = analysis_results['bond_importance']
        
        # 创建子图
        fig = plt.figure(figsize=(20, 6))
        
        # 1. 原子梯度可视化（使用原子级别的信息）
        ax1 = fig.add_subplot(131, projection='3d')
        grad_magnitude = tf.norm(atomic_grads, axis=-1)
        scatter1 = ax1.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                              c=grad_magnitude, cmap='Blues', s=100, alpha=0.7)
        
        # 标注重要的原子（梯度最大的前5个）
        grad_magnitude_np = grad_magnitude.numpy()
        top_atoms = np.argsort(grad_magnitude_np)[-5:]  # 前5个最重要的原子
        for i, atom_idx in enumerate(top_atoms):
            ax1.text(coords[atom_idx, 0], coords[atom_idx, 1], coords[atom_idx, 2], 
                    f'{atomic_nums[atom_idx]}', fontsize=8, color='red', weight='bold')
        
        ax1.set_title('原子梯度大小\n(原子位置变化对预测的影响程度)')
        ax1.set_xlabel('X 坐标')
        ax1.set_ylabel('Y 坐标')
        ax1.set_zlabel('Z 坐标')
        plt.colorbar(scatter1, ax=ax1, shrink=0.5)
        
        # 2. 键重要性可视化（直接显示键的重要性）
        ax2 = fig.add_subplot(132, projection='3d')
        
        # 先显示所有原子（灰色）
        ax2.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                   c='lightgray', s=50, alpha=0.3)
        
        # 显示所有键，用颜色表示重要性
        bond_indices = analysis_results['bond_indices']
        importance = bond_importance['importance']
        
        # 归一化重要性用于颜色映射
        importance_norm = importance / (tf.reduce_max(importance) + 1e-8)
        
        # 绘制所有键，用颜色表示重要性
        for i, (src, dst) in enumerate(zip(bond_indices[0], bond_indices[1])):
            src_coord = coords[src]
            dst_coord = coords[dst]
            bond_importance_val = float(importance_norm[i])
            
            # 用颜色表示重要性
            ax2.plot([src_coord[0], dst_coord[0]], 
                    [src_coord[1], dst_coord[1]], 
                    [src_coord[2], dst_coord[2]], 
                    color=plt.cm.Reds(bond_importance_val), 
                    linewidth=2, alpha=0.7)
        
        ax2.set_title('键重要性可视化\n(颜色强度 = 键重要性)')
        ax2.set_xlabel('X 坐标')
        ax2.set_ylabel('Y 坐标')
        ax2.set_zlabel('Z 坐标')
        
        # 添加颜色条
        sm = plt.cm.ScalarMappable(cmap=plt.cm.Reds, norm=plt.Normalize(vmin=0, vmax=1))
        sm.set_array([])
        plt.colorbar(sm, ax=ax2, shrink=0.5, label='键重要性')
        
        # 3. 重要键连接可视化
        ax3 = fig.add_subplot(133, projection='3d')
        scatter3 = ax3.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                              c='lightgray', s=50, alpha=0.3)
        
        # 绘制重要的键
        threshold = float(tf.reduce_mean(importance)) * 1.5
        important_bonds = 0
        labeled_atoms = set()  # 记录已标注的原子，避免重复标注
        
        for i, (src, dst) in enumerate(zip(bond_indices[0], bond_indices[1])):
            if importance[i] > threshold:
                src_coord = coords[src]
                dst_coord = coords[dst]
                line_width = min(float(importance[i]) * 0.1, 5)  # 限制线宽
                ax3.plot([src_coord[0], dst_coord[0]], 
                        [src_coord[1], dst_coord[1]], 
                        [src_coord[2], dst_coord[2]], 
                        'r-', linewidth=line_width, alpha=0.7)
                
                # 标注重要键两端的原子序数
                src_atomic_num = atomic_nums[src]
                dst_atomic_num = atomic_nums[dst]
                
                # 标注源原子（如果还没标注过）
                if src not in labeled_atoms:
                    ax3.text(src_coord[0], src_coord[1], src_coord[2], 
                            f'{src_atomic_num}', fontsize=8, color='darkred', 
                            weight='bold', ha='center', va='center')
                    labeled_atoms.add(src)
                
                # 标注目标原子（如果还没标注过）
                if dst not in labeled_atoms:
                    ax3.text(dst_coord[0], dst_coord[1], dst_coord[2], 
                            f'{dst_atomic_num}', fontsize=8, color='darkred', 
                            weight='bold', ha='center', va='center')
                    labeled_atoms.add(dst)
                
                important_bonds += 1
        
        ax3.set_title(f'重要化学键\n(显示 {important_bonds} 个最重要的键)\n红色数字: 原子序数')
        ax3.set_xlabel('X 坐标')
        ax3.set_ylabel('Y 坐标')
        ax3.set_zlabel('Z 坐标')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"可视化结果已保存到: {save_path}")
        
        if show_plot:
            plt.show()
        else:
            plt.close()
    
    def generate_summary_report(self, analysis_results, molecule_index=0):
        """生成可解释性分析摘要报告"""
        rbf_attention = analysis_results['rbf_attention']
        atomic_grads = analysis_results['atomic_gradients']
        bond_importance = analysis_results['bond_importance']
        molecular_contribution = analysis_results['molecular_contribution']
        
        # 计算统计信息
        grad_magnitude = tf.norm(atomic_grads, axis=-1)
        
        report = {
            'molecule_index': molecule_index,
            'prediction': float(molecular_contribution),
            'rbf_attention_stats': {
                'mean': float(tf.reduce_mean(rbf_attention)),
                'std': float(tf.math.reduce_std(rbf_attention)),
                'min': float(tf.reduce_min(rbf_attention)),
                'max': float(tf.reduce_max(rbf_attention))
            },
            'atomic_gradient_stats': {
                'mean': float(tf.reduce_mean(grad_magnitude)),
                'std': float(tf.math.reduce_std(grad_magnitude)),
                'min': float(tf.reduce_min(grad_magnitude)),
                'max': float(tf.reduce_max(grad_magnitude))
            },
            'bond_importance_stats': {
                'mean': float(tf.reduce_mean(bond_importance['importance'])),
                'std': float(tf.math.reduce_std(bond_importance['importance'])),
                'min': float(tf.reduce_min(bond_importance['importance'])),
                'max': float(tf.reduce_max(bond_importance['importance']))
            },
            'num_atoms': int(tf.shape(atomic_grads)[0]),
            'num_bonds': int(tf.shape(bond_importance['importance'])[0])
        }
        
        return report


In [ ]:
### 可解释性分析辅助函数
def get_element_symbol(atomic_number):
    """获取原子序数对应的元素符号"""
    elements = {
        1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F', 13: 'Al', 16: 'S', 17: 'Cl', 35: 'Br'
    }
    return elements.get(atomic_number, f'Z{atomic_number}')

def safe_explain_single_molecule(model, data_container, molecule_index=0):
    """安全的单个分子可解释性分析"""
    try:
        print(f"🔍 分析分子 {molecule_index} 的可解释性...")
        
        # 确保索引是Python int类型
        molecule_index = int(molecule_index)
        
        # 检查索引是否在有效范围内
        if molecule_index >= len(data_container):
            raise ValueError(f"分子索引 {molecule_index} 超出范围 [0, {len(data_container)-1}]")
        
        # 获取分子数据
        inputs = data_container[molecule_index]
        
        # 检查输入数据是否完整
        required_keys = ['R', 'Z', 'idnb_i', 'idnb_j']
        missing_keys = [key for key in required_keys if key not in inputs]
        if missing_keys:
            raise ValueError(f"输入数据缺少必要的键: {missing_keys}")
        
        # 创建可解释性分析器
        explainer = DimeNetExplainability(model, data_container)
        
        # 进行可解释性分析
        results = explainer.analyze_molecule(inputs, target_class=0)
        
        # 生成摘要报告
        report = explainer.generate_summary_report(results, molecule_index)
        
        # 打印摘要
        print(f"\n📊 分子 {molecule_index} 可解释性分析摘要:")
        print(f"  预测值: {report['prediction']:.4f}")
        print(f"  原子数量: {report['num_atoms']}")
        print(f"  键数量: {report['num_bonds']}")
        print(f"  RBF注意力权重范围: [{report['rbf_attention_stats']['min']:.4f}, {report['rbf_attention_stats']['max']:.4f}]")
        print(f"  原子梯度大小范围: [{report['atomic_gradient_stats']['min']:.4f}, {report['atomic_gradient_stats']['max']:.4f}]")
        print(f"  键重要性范围: [{report['bond_importance_stats']['min']:.4f}, {report['bond_importance_stats']['max']:.4f}]")
        
        return results, report
        
    except Exception as e:
        print(f"❌ 分子 {molecule_index} 分析失败: {e}")
        print(f"  错误类型: {type(e).__name__}")
        return None, None

def explain_single_molecule(model, data_container, molecule_index=0, save_dir=None):
    """解释单个分子的预测（使用安全方法）"""
    return safe_explain_single_molecule(model, data_container, molecule_index)

def find_top_predictions(predictions, true_values, molecule_indices, top_k=10):
    """找到预测最准确的前k个分子（按绝对误差最小排序）"""
    # 计算绝对误差
    absolute_errors = np.abs(predictions - true_values)
    
    # 将预测值、真实值、绝对误差和分子索引配对
    pred_pairs = list(zip(predictions, true_values, absolute_errors, molecule_indices))
    
    # 按绝对误差升序排序（误差越小越好）
    pred_pairs.sort(key=lambda x: x[2])
    
    # 返回前k个
    top_predictions = pred_pairs[:top_k]
    
    print(f"\n🏆 预测最准确的前{top_k}个分子 (按绝对误差排序):")
    print("=" * 80)
    for i, (pred_value, true_value, abs_error, mol_idx) in enumerate(top_predictions):
        print(f"  {i+1:2d}. 分子索引: {mol_idx:4d} | 真实值: {true_value:.4f} | 预测值: {pred_value:.4f} | 绝对误差: {abs_error:.4f}")
    print("=" * 80)
    
    # 额外统计信息
    all_errors = [error for _, _, error, _ in pred_pairs]
    print(f"\n📊 误差统计信息:")
    print(f"  最小绝对误差: {min(all_errors):.6f}")
    print(f"  最大绝对误差: {max(all_errors):.6f}")
    print(f"  平均绝对误差: {np.mean(all_errors):.6f}")
    print(f"  误差标准差: {np.std(all_errors):.6f}")
    print(f"  总分子数: {len(all_errors)}")
    
    return top_predictions

def find_worst_predictions(predictions, true_values, molecule_indices, worst_k=10):
    """找到预测最不准确的前k个分子（按绝对误差最大排序）"""
    # 计算绝对误差
    absolute_errors = np.abs(predictions - true_values)
    
    # 将预测值、真实值、绝对误差和分子索引配对
    pred_pairs = list(zip(predictions, true_values, absolute_errors, molecule_indices))
    
    # 按绝对误差降序排序（误差越大越差）
    pred_pairs.sort(key=lambda x: x[2], reverse=True)
    
    # 返回前k个
    worst_predictions = pred_pairs[:worst_k]
    
    print(f"\n💥 预测最不准确的前{worst_k}个分子 (按绝对误差排序):")
    print("=" * 80)
    for i, (pred_value, true_value, abs_error, mol_idx) in enumerate(worst_predictions):
        print(f"  {i+1:2d}. 分子索引: {mol_idx:4d} | 真实值: {true_value:.4f} | 预测值: {pred_value:.4f} | 绝对误差: {abs_error:.4f}")
    print("=" * 80)
    
    return worst_predictions


In [ ]:
### 单个分子可解释性分析示例
print("🔍 开始单个分子可解释性分析...")

# 选择第一个测试分子进行分析
test_molecule_index = 0
print(f"分析分子索引: {test_molecule_index}")

# 进行可解释性分析
try:
    results, report = explain_single_molecule(model, data_container, test_molecule_index)
    print("✅ 单个分子可解释性分析完成！")
    
    # 显示分析结果摘要
    print(f"\n📊 分析结果摘要:")
    print(f"  分子索引: {report['molecule_index']}")
    print(f"  预测值: {report['prediction']:.6f}")
    print(f"  原子数量: {report['num_atoms']}")
    print(f"  键数量: {report['num_bonds']}")
    
except Exception as e:
    print(f"❌ 单个分子分析失败: {e}")
    print("请检查模型和数据是否正确加载")


In [ ]:
### 可解释性可视化
print("🎨 开始生成可解释性可视化...")

# 创建可解释性分析器
explainer = DimeNetExplainability(model, data_container)

# 对第一个分子进行可视化
try:
    print(f"正在为分子 {test_molecule_index} 生成可视化...")
    
    # 生成可视化图片
    explainer.visualize_explanations(results, test_molecule_index, show_plot=True)
    
    print("✅ 可解释性可视化完成！")
    print("\n🔍 可视化说明:")
    print("  左图 (蓝色): 原子梯度大小 - 显示每个原子位置变化对预测的影响程度")
    print("  中图 (红色): 键重要性 - 显示每个化学键的重要性，颜色越深越重要")
    print("  右图 (红色线条): 重要化学键 - 显示对预测最重要的化学键连接")
    print("  红色数字: 原子序数，线条越粗表示键越重要")
    
except Exception as e:
    print(f"❌ 可视化生成失败: {e}")
    print("请检查分析结果是否正确生成")


In [ ]:
### 批量分子可解释性分析
print("🔍 开始批量分子可解释性分析...")

# 从测试集预测结果中找到最佳和最差的预测
print("📊 分析测试集预测结果...")

# 使用已有的预测结果和真实值
test_predictions = preds_total[:, 0]  # 取第一个目标
test_true_values = targets_total[:, 0]  # 取第一个目标的真实值
test_indices = np.arange(len(test_predictions))

# 找到预测最准确的前5个分子（按绝对误差最小排序）
top_5_molecules = find_top_predictions(test_predictions, test_true_values, test_indices, top_k=5)

# 找到预测最不准确的前5个分子（按绝对误差最大排序）
worst_5_molecules = find_worst_predictions(test_predictions, test_true_values, test_indices, worst_k=5)

print(f"\n🎯 将分析前5个最准确预测分子和前5个最不准确预测分子")


In [ ]:
### 分析前5个最准确预测分子
print("🏆 开始分析前5个最准确预测分子...")

# 创建保存目录
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f"explanations_{timestamp}"
os.makedirs(save_dir, exist_ok=True)

successful_analyses = 0
failed_analyses = 0
all_results = []

for i, (pred_value, true_value, abs_error, mol_idx) in enumerate(top_5_molecules):
    try:
        print(f"\n[{i+1}/5] 分析第{i+1}名最准确预测分子:")
        print(f"  分子索引: {mol_idx}")
        print(f"  真实值: {true_value:.4f}")
        print(f"  预测值: {pred_value:.4f}")
        print(f"  绝对误差: {abs_error:.4f}")
        
        # 使用修复后的方法进行可解释性分析
        results, report = safe_explain_single_molecule(model, data_container, mol_idx)
        
        if results is not None and report is not None:
            all_results.append((mol_idx, results, report, pred_value, true_value, abs_error))
            successful_analyses += 1
            
            # 生成可视化图片
            save_path = os.path.join(save_dir, f"top_{i+1}_molecule_{mol_idx}_pred_{pred_value:.4f}.png")
            explainer.visualize_explanations(results, mol_idx, save_path, show_plot=False)
            
            print(f"✅ 第{i+1}名分子 {mol_idx} 分析和可视化完成")
        else:
            failed_analyses += 1
            print(f"❌ 第{i+1}名分子 {mol_idx} 分析失败")
        
    except Exception as e:
        failed_analyses += 1
        print(f"❌ 第{i+1}名分子 {mol_idx} 分析失败: {e}")
        continue

print(f"\n📊 最准确预测分子分析统计:")
print(f"  成功分析: {successful_analyses} 个分子")
print(f"  失败分析: {failed_analyses} 个分子")
print(f"  成功率: {100*successful_analyses/5:.1f}%")
print(f"  结果保存在: {save_dir}")


In [ ]:
### 分析前5个最不准确预测分子
print("💥 开始分析前5个最不准确预测分子...")

worst_successful_analyses = 0
worst_failed_analyses = 0
worst_all_results = []

for i, (pred_value, true_value, abs_error, mol_idx) in enumerate(worst_5_molecules):
    try:
        print(f"\n[{i+1}/5] 分析第{i+1}名最不准确预测分子:")
        print(f"  分子索引: {mol_idx}")
        print(f"  真实值: {true_value:.4f}")
        print(f"  预测值: {pred_value:.4f}")
        print(f"  绝对误差: {abs_error:.4f}")
        
        # 使用修复后的方法进行可解释性分析
        results, report = safe_explain_single_molecule(model, data_container, mol_idx)
        
        if results is not None and report is not None:
            worst_all_results.append((mol_idx, results, report, pred_value, true_value, abs_error))
            worst_successful_analyses += 1
            
            # 生成可视化图片
            save_path = os.path.join(save_dir, f"worst_{i+1}_molecule_{mol_idx}_pred_{pred_value:.4f}.png")
            explainer.visualize_explanations(results, mol_idx, save_path, show_plot=False)
            
            print(f"✅ 第{i+1}名最不准确分子 {mol_idx} 分析和可视化完成")
        else:
            worst_failed_analyses += 1
            print(f"❌ 第{i+1}名最不准确分子 {mol_idx} 分析失败")
        
    except Exception as e:
        worst_failed_analyses += 1
        print(f"❌ 第{i+1}名最不准确分子 {mol_idx} 分析失败: {e}")
        continue

print(f"\n📊 最不准确预测分子分析统计:")
print(f"  成功分析: {worst_successful_analyses} 个分子")
print(f"  失败分析: {worst_failed_analyses} 个分子")
print(f"  成功率: {100*worst_successful_analyses/5:.1f}%")


In [ ]:
### 保存分析结果到CSV文件
print("💾 保存分析结果...")

# 创建结果数据
results_data = []

# 添加最准确预测分子结果
for i, (pred_value, true_value, abs_error, mol_idx) in enumerate(top_5_molecules):
    results_data.append({
        '排名': i + 1,
        '分子索引': mol_idx,
        '真实值': true_value,
        '预测值': pred_value,
        '绝对误差': abs_error,
        '类型': '最准确预测',
        '分析状态': '成功' if i < successful_analyses else '失败'
    })

# 添加最不准确预测分子结果
for i, (pred_value, true_value, abs_error, mol_idx) in enumerate(worst_5_molecules):
    results_data.append({
        '排名': i + 1,
        '分子索引': mol_idx,
        '真实值': true_value,
        '预测值': pred_value,
        '绝对误差': abs_error,
        '类型': '最不准确预测',
        '分析状态': '成功' if i < worst_successful_analyses else '失败'
    })

# 保存到CSV文件
df = pd.DataFrame(results_data)
csv_path = os.path.join(save_dir, "explainability_analysis_results.csv")
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f"📊 分析结果已保存到: {csv_path}")

# 显示最终统计
print(f"\n🎉 批量可解释性分析完成！")
print(f"📁 所有结果保存在: {save_dir}")
print(f"📊 总体分析统计:")
print(f"  最准确预测 - 成功分析: {successful_analyses}/5 个分子")
print(f"  最不准确预测 - 成功分析: {worst_successful_analyses}/5 个分子")
print(f"  总成功率: {100*(successful_analyses + worst_successful_analyses)/10:.1f}%")
print(f"  生成的可视化图片数量: {successful_analyses + worst_successful_analyses} 张")


## 可解释性分析功能说明

本notebook已成功集成了DimeNet++模型的可解释性分析功能，包括：

### 🔍 核心功能
1. **DimeNetExplainability类**: 基于RBF注意力机制的可解释性分析器
2. **原子梯度分析**: 计算每个原子位置变化对预测的影响程度
3. **键重要性分析**: 识别对预测最重要的化学键
4. **RBF注意力权重**: 提取模型内部的注意力机制

### 📊 分析方法
- **原子级分析**: 通过梯度计算识别重要原子
- **键级分析**: 结合RBF注意力和梯度分析重要化学键
- **分子级分析**: 整体分子对预测的贡献

### 🎨 可视化功能
- **3D原子梯度图**: 蓝色表示原子梯度大小
- **键重要性图**: 红色表示键的重要性
- **重要化学键图**: 突出显示最重要的化学键连接

### 📈 批量分析
- 自动识别预测最好和最差的分子
- 批量生成可解释性分析报告
- 保存可视化图片和分析结果

### 💾 输出文件
- 可视化图片: PNG格式的3D分子结构图
- 分析报告: CSV格式的详细分析结果
- 统计信息: 成功率和分析质量指标

这些功能帮助理解DimeNet++模型的预测机制，识别对预测结果最重要的原子和化学键。


In [ ]:
### 调试数据容器访问
print("🔍 调试数据容器访问...")

# 检查数据容器长度
print(f"数据容器总长度: {len(data_container)}")

# 测试访问第一个分子
try:
    test_inputs = data_container[0]
    print(f"✅ 成功访问分子 0")
    print(f"  输入键: {list(test_inputs.keys())}")
    print(f"  原子数量: {len(test_inputs['Z'])}")
    print(f"  坐标形状: {test_inputs['R'].shape}")
    print(f"  键数量: {len(test_inputs['idnb_i'])}")
except Exception as e:
    print(f"❌ 访问分子 0 失败: {e}")

# 测试访问分子 265（第一个最佳预测分子）
try:
    test_inputs_265 = data_container[265]
    print(f"✅ 成功访问分子 265")
    print(f"  输入键: {list(test_inputs_265.keys())}")
    print(f"  原子数量: {len(test_inputs_265['Z'])}")
    print(f"  坐标形状: {test_inputs_265['R'].shape}")
    print(f"  键数量: {len(test_inputs_265['idnb_i'])}")
except Exception as e:
    print(f"❌ 访问分子 265 失败: {e}")
    print(f"  错误类型: {type(e)}")
    print(f"  错误详情: {str(e)}")


In [ ]:
### 测试修复后的方法
print("🔧 测试修复后的方法...")

# 测试修复后的方法
test_molecule_index = 0
print(f"测试分子索引: {test_molecule_index}")
results, report = safe_explain_single_molecule(model, data_container, test_molecule_index)

if results is not None:
    print("✅ 修复后的方法工作正常！")
else:
    print("❌ 修复后的方法仍然失败")


In [ ]:
### 简化的单分子测试
print("🧪 进行简化的单分子测试...")

# 只测试第一个分子
test_molecule_idx = 0
print(f"测试分子索引: {test_molecule_idx}")

try:
    # 使用修复后的方法
    results, report = safe_explain_single_molecule(model, data_container, test_molecule_idx)
    
    if results is not None and report is not None:
        print("✅ 单分子测试成功！")
        
        # 尝试生成可视化
        try:
            explainer = DimeNetExplainability(model, data_container)
            explainer.visualize_explanations(results, test_molecule_idx, show_plot=True)
            print("✅ 可视化生成成功！")
        except Exception as viz_e:
            print(f"❌ 可视化生成失败: {viz_e}")
    else:
        print("❌ 单分子测试失败")
        
except Exception as e:
    print(f"❌ 单分子测试异常: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
### 测试修改后的排序逻辑
print("🧪 测试修改后的排序逻辑...")

# 使用测试数据验证新的排序逻辑
print("📊 验证排序逻辑是否正确...")

# 创建一些测试数据
test_pred = np.array([100, 200, 300, 400, 500])
test_true = np.array([110, 190, 310, 380, 520])  # 故意设置不同的真实值
test_indices = np.arange(len(test_pred))

# 计算绝对误差
test_errors = np.abs(test_pred - test_true)
print(f"测试数据:")
print(f"  预测值: {test_pred}")
print(f"  真实值: {test_true}")
print(f"  绝对误差: {test_errors}")

# 测试最准确预测（误差最小）
print(f"\n🔍 测试最准确预测排序:")
top_test = find_top_predictions(test_pred, test_true, test_indices, top_k=3)

# 测试最不准确预测（误差最大）
print(f"\n🔍 测试最不准确预测排序:")
worst_test = find_worst_predictions(test_pred, test_true, test_indices, worst_k=3)

print(f"\n✅ 排序逻辑测试完成！")
print(f"  最准确预测应该按误差升序排列")
print(f"  最不准确预测应该按误差降序排列")
